# Valuation Model: Predicting Target Price (TP) and Strong Buy Below (SBB)

Trains an ML model to predict two analyst price targets for any stock:
- **Target Price (TP)**: The analyst's 12-18 month fair value estimate  
- **Strong Buy Below (SBB)**: The entry price with a margin of safety to TP

**Training data**: All `data/GP/*.xlsx` and `data/GP/*.xls` analyst spreadsheets (~9,200 labeled rows, 286 tickers, 2012-2026).  
**Features**: Spreadsheet-native fundamentals (date-aligned) + pe_stats historical metrics.  
**Approach**: Predict `tp_ratio = TP / current_price`, then derive `SBB = TP × calibrated_ratio` (~0.582).  
**To add more training data**: drop additional `GP*.xlsx` files into `data/GP/` and re-run.

---
## Cell 1 — Imports

In [ ]:
import sys, glob, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import duckdb
from dotenv import load_dotenv
from tqdm.auto import tqdm
import os

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env', override=True)

from historic_fundamentals import get_pe_stats
from historic_fundamentals.db import HistoricFundamentalsDB

print('Python:', sys.version[:20])
print('ROOT:', ROOT)

---
## Cell 2 — Configuration

All tunable parameters are here. Adjust and re-run without touching downstream cells.

In [ ]:
DATA_DIR   = ROOT / 'data' / 'GP'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

SPREADSHEET_PATTERN = str(DATA_DIR / '*.xlsx'), str(DATA_DIR / '*.xls')

HF_DB_PATH = os.getenv('HF_DB_PATH', str(ROOT / 'data' / 'historic_fundamentals.duckdb'))

CV_FOLDS     = 5
RANDOM_STATE = 42
N_ESTIMATORS = 300          # GBM trees (more = slower but better)
MIN_FEATURE_COVERAGE = 0.40 # drop features with fewer non-null values than this

MODEL_FILE    = MODELS_DIR / 'tp_model.joblib'
FEATURES_FILE = MODELS_DIR / 'tp_features.joblib'
CALIB_FILE    = MODELS_DIR / 'sbb_calibration.joblib'

# Reference date for computing spreadsheet vintage (months ago)
TODAY = pd.Timestamp.now()

print('Spreadsheet dir:', DATA_DIR)
print('HF DB:', HF_DB_PATH)
print('Model output:', MODELS_DIR)

---
## Cell 3 — Multi-Format Spreadsheet Parser

Handles all 7+ format variants found in the GP spreadsheet history (16–25 columns, 2012–2026).
Detects header row by keyword search, then locates each data column by keyword + value validation.

**Key disambiguation rules:**
- SBB vs `% below`: validated by checking median absolute value > \$2
- TP vs current price: TP always appears *after* SBB in the column order
- `yld` = dividend yield (exact match); `yield` = FCF yield (different column in newer files)

In [ ]:
def _val_median(data_df, col, n=15):
    vals = pd.to_numeric(data_df.iloc[:n, col], errors='coerce').dropna()
    return float(vals.abs().median()) if len(vals) > 0 else 0.0

def _find_validated(keywords, data_df, headers, exclude_kw=None, min_val=2.0):
    """First column matching any keyword whose data values have abs-median > min_val."""
    for i, h in enumerate(headers):
        if exclude_kw and any(ex in h for ex in exclude_kw):
            continue
        if any(kw in h for kw in keywords):
            if _val_median(data_df, i) > min_val:
                return i
    return None

def _col(keywords, headers, exclude_kw=None):
    """First column index matching any keyword."""
    for i, h in enumerate(headers):
        if exclude_kw and any(ex in h for ex in exclude_kw):
            continue
        if any(kw in h for kw in keywords):
            return i
    return None

def _parse_sheet_date(stem):
    digits = stem[2:]
    for fmt in ('%m%d%y', '%Y%m%d'):
        try:
            return pd.to_datetime(digits, format=fmt)
        except ValueError:
            pass
    return pd.NaT

def load_spreadsheet(path):
    """
    Parse one GP spreadsheet into a tidy DataFrame.
    Returns (df, error_msg). df is empty on failure.
    """
    df_raw = pd.read_excel(path, header=None)
    rows_lower = [
        [str(v).strip().lower() for v in df_raw.iloc[i]]
        for i in range(min(8, len(df_raw)))
    ]

    # Find header row: prefer row with 'symbol', fall back to row with 'buy'/'target'
    header_row = None
    for i, row in enumerate(rows_lower):
        if any(r in ('symbol', 'ticker', 'sym') for r in row):
            header_row = i
            break
    if header_row is None:
        for i, row in enumerate(rows_lower):
            if any(r in ('buy', 'target', 'target price') for r in row):
                header_row = i
                break
    if header_row is None:
        return pd.DataFrame(), 'no header row found'

    hl = rows_lower[header_row]
    data = df_raw.iloc[header_row + 1:].reset_index(drop=True)

    sym_col = _col(['symbol', 'ticker'], hl)
    if sym_col is None:
        sym_col = 0  # oldest formats: ticker is always col 0

    # SBB: first col with keyword AND absolute price values (> $2)
    sbb_col = _find_validated(
        ['strong buy below', 'strong buy', 'buy below', 'buy', 'below'],
        data, hl, min_val=2.0)

    # Target price: 'target' keyword (validated), then first 'price' col AFTER sbb_col
    tp_col = None
    for i, h in enumerate(hl):
        if any(kw in h for kw in ['target price', 'target']):
            if _val_median(data, i) > 2.0:
                tp_col = i
                break
    if tp_col is None and sbb_col is not None:
        for i, h in enumerate(hl):
            if 'price' in h and i > sbb_col and _val_median(data, i) > 2.0:
                tp_col = i
                break
    if tp_col is None:
        price_cols = [i for i, h in enumerate(hl) if 'price' in h]
        for i in reversed(price_cols):
            if _val_median(data, i) > 2.0:
                tp_col = i
                break

    # Current price: first 'price' col BEFORE sbb_col; fallback sym_col+2
    price_col = None
    for i, h in enumerate(hl):
        excl = ('target', 'strong', 'above', 'below', 'buy')
        if 'price' in h and not any(ex in h for ex in excl):
            if sbb_col is None or i < sbb_col:
                if _val_median(data, i) > 0.5:
                    price_col = i
                    break
    if price_col is None:
        price_col = sym_col + 2

    pe_col   = _col(['p/e', 'p_e'], hl)
    ps_col   = _col(['p/s', 'p_s'], hl)
    ptbv_col = _col(['p/tbv', 'ptbv', 'tbv'], hl)
    ev_col   = _col(['ebitda'], hl)
    yld_col  = _col(['yld'], hl)                   # dividend yield ('yld' exact)
    fcf_col  = _col(['fcf'], hl)                   # FCF yield
    if fcf_col is None:                             # newer files: FCF col labeled 'yield'
        yield_cols = [i for i, h in enumerate(hl) if 'yield' in h and i != yld_col]
        if yield_cols:
            fcf_col = yield_cols[-1]
    roce_col = _col(['roce', 'roc'], hl)
    cap_col  = _col(['marcap', 'market cap', 'cap'], hl)
    tps_col  = _col(['tps'], hl)
    sheet_date = _parse_sheet_date(Path(path).stem)

    def _get(row, c):
        if c is None:
            return np.nan
        v = row.iloc[c]
        if pd.isna(v) or str(v).strip().lower() in ('nan', 'nmf', '', 'n/a', '#n/a'):
            return np.nan
        try:
            return float(v)
        except (ValueError, TypeError):
            return np.nan

    rows = []
    for _, row in data.iterrows():
        ticker = str(row.iloc[sym_col]).strip()
        if not ticker or ticker.lower() in ('nan', 'symbol', 'ticker', '', 'none', 'common stock'):
            continue
        if len(ticker) > 10:   # guard against long strings that are headers/noise
            continue
        mc = _get(row, cap_col)
        rows.append({
            'ticker':           ticker,
            'sheet_price':      _get(row, price_col),
            'sbb':              _get(row, sbb_col),
            'target_price':     _get(row, tp_col),
            'sheet_pe':         _get(row, pe_col),
            'sheet_ps':         _get(row, ps_col),
            'sheet_ptbv':       _get(row, ptbv_col),
            'sheet_ev_ebitda':  _get(row, ev_col),
            'sheet_fcf_yield':  _get(row, fcf_col),
            'sheet_div_yield':  _get(row, yld_col),
            'sheet_roce':       _get(row, roce_col),
            'log_sheet_mktcap': np.log1p(mc) if mc and mc > 0 else np.nan,
            'sheet_tps':        _get(row, tps_col),
            'sheet_date':       sheet_date,
            'source_file':      Path(path).name,
        })
    return pd.DataFrame(rows), None

print('Parser defined.')

---
## Cell 4 — Load All Spreadsheets

Parses every `GP*.xlsx` / `GP*.xls` file in `data/GP/`.
Any file that fails or yields 0 valid rows is reported but does not abort the run.

In [ ]:
files = sorted(glob.glob(SPREADSHEET_PATTERN[0]) + glob.glob(SPREADSHEET_PATTERN[1]))
print(f'Found {len(files)} spreadsheet(s) in {DATA_DIR}')

all_dfs, parse_errors = [], []
for path in tqdm(files, desc='Parsing spreadsheets', unit='file'):
    df, err = load_spreadsheet(path)
    if err:
        parse_errors.append((Path(path).name, err))
        continue
    valid = df.dropna(subset=['ticker', 'sheet_price', 'sbb', 'target_price'])
    valid = valid[(valid['sheet_price'] > 0) & (valid['sbb'] > 0) & (valid['target_price'] > 0)]
    if len(valid) == 0:
        parse_errors.append((Path(path).name, '0 valid rows after filter'))
    else:
        all_dfs.append(valid)

if parse_errors:
    print(f'\nWarnings ({len(parse_errors)} files):')
    for name, msg in parse_errors:
        print(f'  {name}: {msg}')

labels = pd.concat(all_dfs, ignore_index=True)
print(f'\nTotal labeled rows:  {len(labels):,}')
print(f'Files contributing:  {len(all_dfs)}')
print(f'Unique tickers:      {labels["ticker"].nunique()}')
print(f'Date range:          {labels["sheet_date"].min().date()} to {labels["sheet_date"].max().date()}')
print()
print('Feature coverage (fraction non-null):')
for col in ['sheet_pe','sheet_ps','sheet_ptbv','sheet_ev_ebitda',
            'sheet_fcf_yield','sheet_div_yield','sheet_roce','sheet_tps']:
    print(f'  {col:22}: {labels[col].notna().mean():.0%}')

---
## Cell 5 — Compute Target Variables

We predict **ratios** relative to current price — they generalize better across stocks at different price levels.

- `tp_ratio = TP / current_price` — upside to fair value  
- `sbb_tp_ratio = SBB / TP` — used later to calibrate SBB from TP (~0.582, very consistent)

In [ ]:
labels['tp_ratio']     = labels['target_price'] / labels['sheet_price']
labels['sbb_tp_ratio'] = labels['sbb']          / labels['target_price']

# Sanity filters
before = len(labels)
labels = labels[(labels['tp_ratio'] >= 0.8) & (labels['tp_ratio'] <= 10.0)]
labels = labels[(labels['sbb_tp_ratio'] > 0) & (labels['sbb_tp_ratio'] < 1.5)]
print(f'Rows after sanity filter: {len(labels):,} (dropped {before - len(labels)})')
print()
print('Target variable summary:')
print(labels[['tp_ratio', 'sbb_tp_ratio']].describe().round(3))

---
## Cell 6 — Load pe_stats Features from Database

For each ticker in the training data we pull current pe_stats features.

**Temporal note**: pe_stats captures current values, but training labels span 2012–2026.
For the oldest rows, structural metrics (pe_lt_median, rev_cagr, roic) are approximate.
GradientBoosting handles noisy features well; the spreadsheet-native features (Cell 7)
provide the date-aligned anchor.

In [ ]:
all_tickers = labels['ticker'].unique().tolist()
pe_stats = get_pe_stats(all_tickers)

print(f'Tickers in labels:   {len(all_tickers)}')
print(f'Tickers in pe_stats: {len(pe_stats)}')

missing = set(all_tickers) - set(pe_stats['ticker'])
if missing:
    print(f'\nNot in pe_stats ({len(missing)} tickers — likely delisted or not yet imported):')
    print(' ', sorted(missing)[:30])

# Merge labels with pe_stats
train_df = labels.merge(pe_stats, on='ticker', how='inner')
print(f'\nTraining rows after merge: {len(train_df):,}')
print(f'Unique tickers in training: {train_df["ticker"].nunique()}')

---
## Cell 7 — Feature Engineering

Two functions are defined:

- `build_features(df, mode='train')` — builds the feature matrix.  
  In `train` mode uses spreadsheet-native columns (date-aligned).  
  In `predict` mode maps current pe_stats columns to the same feature names.

Feature groups:
1. **Valuation level** — current multiples (PE, PS, P/TBV, EV/EBITDA, FCF yield)
2. **Mean-reversion signal** — current / historical-median (premium or discount vs own history)
3. **Growth** — revenue, earnings, FCF CAGRs and NTM estimates
4. **Quality** — ROIC, ROE, ROA (long-term medians)
5. **Size + income** — log(market cap), dividend yield
6. **Vintage** — months since spreadsheet was generated (proxy for pe_stats drift for older rows)

In [ ]:
def _safe_ratio(a, b, clip=20.0):
    return (a / b.replace(0, np.nan)).clip(-clip, clip)

def build_features(df, mode='train'):
    """
    mode='train'  : use spreadsheet columns (sheet_pe, sheet_ps, ...) for valuation features.
    mode='predict': use pe_stats current columns (current_pe, current_ps, ...) instead.
    Both modes produce identically-named output columns.
    """
    f = pd.DataFrame(index=df.index)

    # ── 1. Valuation level ───────────────────────────────────────────────────
    if mode == 'train':
        f['f_pe']        = df['sheet_pe']
        f['f_ps']        = df['sheet_ps']
        f['f_ptbv']      = df['sheet_ptbv']
        f['f_ev_ebitda'] = df['sheet_ev_ebitda']
        f['f_fcf_yield'] = df['sheet_fcf_yield']
        f['f_div_yield'] = df['sheet_div_yield']
        f['f_roce']      = df['sheet_roce']
        f['f_log_mktcap']= df['log_sheet_mktcap']
        f['f_tps']       = df.get('sheet_tps', pd.Series(np.nan, index=df.index))
        # Vintage: months since spreadsheet was generated (captures pe_stats temporal drift)
        f['f_vintage_months'] = (
            (TODAY - df['sheet_date']).dt.days / 30.44
        ).clip(0, 200)
    else:  # predict mode: map pe_stats current columns
        f['f_pe']        = df['current_pe']
        f['f_ps']        = df['current_ps']
        f['f_ptbv']      = df['current_ptbv']
        f['f_ev_ebitda'] = df['current_evebitda']
        f['f_fcf_yield'] = df['current_fcf_yield']
        f['f_div_yield'] = df['dividend_yield']
        f['f_roce']      = df['current_roa']   # best available proxy for ROCE
        f['f_log_mktcap']= np.log1p(df['market_cap_b'].clip(lower=0))
        f['f_tps']       = pd.Series(np.nan, index=df.index)  # not available for new tickers
        f['f_vintage_months'] = 0.0            # prediction is always current

    # ── 2. Mean-reversion: current multiple / long-term median ───────────────
    # Values > 1 = stock trades at premium to its own history
    f['f_pe_premium']   = _safe_ratio(f['f_pe'],        df['pe_lt_median'])
    f['f_ps_premium']   = _safe_ratio(f['f_ps'],        df['ps_lt_median'])
    f['f_ptbv_premium'] = _safe_ratio(f['f_ptbv'],      df['ptbv_lt_median'])
    f['f_ev_premium']   = _safe_ratio(f['f_ev_ebitda'], df['evebitda_lt_median'])

    # ── 3. Historical median multiples (the fair-value anchors) ─────────────
    f['f_pe_lt_med']    = df['pe_lt_median']
    f['f_ps_lt_med']    = df['ps_lt_median']
    f['f_pfcf_lt_med']  = df['pfcf_lt_median']
    f['f_ev_lt_med']    = df['evebitda_lt_median']
    f['f_ptbv_lt_med']  = df['ptbv_lt_median']
    f['f_pe_5yr_med']   = df['pe_rolling_5yr_median']
    f['f_ps_5yr_med']   = df['ps_rolling_5yr_median']
    f['f_forward_pe']   = df['forward_pe']

    # ── 4. Growth metrics ────────────────────────────────────────────────────
    f['f_rev_g1']       = df['rev_growth_1yr']
    f['f_rev_c3']       = df['rev_cagr_3yr']
    f['f_rev_c5']       = df['rev_cagr_5yr']
    f['f_rev_ntm']      = df['rev_ntm_growth_est']
    f['f_earn_g1']      = df['earn_growth_1yr']
    f['f_earn_c3']      = df['earn_cagr_3yr']
    f['f_earn_c5']      = df['earn_cagr_5yr']
    f['f_earn_ntm']     = df['earn_ntm_growth_est']
    f['f_fcf_g1']       = df['fcf_growth_1yr']
    f['f_fcf_c3']       = df['fcf_cagr_3yr']
    f['f_fcf_c5']       = df['fcf_cagr_5yr']
    f['f_fcf_margin']   = df['fcf_margin_5yr_median']
    f['f_ebitda_margin']= df['ebitda_margin_5yr_median']

    # ── 5. Quality / profitability ───────────────────────────────────────────
    f['f_roic_lt']      = df['roic_lt_median']
    f['f_roe_lt']       = df['roe_lt_median']
    f['f_roa_lt']       = df['roa_lt_median']
    f['f_months_hist']  = df['months_available']

    return f

features_train = build_features(train_df, mode='train')
print(f'Feature matrix: {features_train.shape[0]:,} rows × {features_train.shape[1]} features')
print('Feature names:', list(features_train.columns))

---
## Cell 8 — EDA: Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title in [
    (axes[0], 'tp_ratio',     'TP / Current Price  (target)'),
    (axes[1], 'sbb_tp_ratio', 'SBB / TP  (calibration constant)'),
]:
    data = train_df[col].dropna()
    ax.hist(data, bins=40, edgecolor='white', color='steelblue', alpha=0.8)
    ax.axvline(data.median(), color='red',    linestyle='--', label=f'Median={data.median():.3f}')
    ax.axvline(data.mean(),   color='orange', linestyle='--', label=f'Mean={data.mean():.3f}')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(train_df[['tp_ratio', 'sbb_tp_ratio']].describe().round(3))

---
## Cell 9 — EDA: Feature Correlation with Target

In [ ]:
target_s = train_df['tp_ratio']
corr_rows = []
for col in features_train.columns:
    mask = features_train[col].notna() & target_s.notna()
    if mask.sum() >= 30:
        r = np.corrcoef(features_train.loc[mask, col], target_s[mask])[0, 1]
        corr_rows.append({'feature': col, 'r': r, 'n': int(mask.sum())})

corr_df = pd.DataFrame(corr_rows).sort_values('r', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, max(5, len(corr_df) * 0.3)))
colors = ['steelblue' if r >= 0 else 'coral' for r in corr_df['r']]
ax.barh(corr_df['feature'], corr_df['r'], color=colors, alpha=0.8, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Pearson r with tp_ratio')
ax.set_title('Feature Correlation with TP / current_price')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 15 features:')
print(corr_df.head(15).to_string(index=False))

---
## Cell 10 — Prepare ML Dataset

Drop low-coverage features, define X and y.  
The pipeline handles median imputation internally so NaN values in X are fine.

In [ ]:
coverage = features_train.notna().mean()
dropped  = coverage[coverage < MIN_FEATURE_COVERAGE].index.tolist()
if dropped:
    print(f'Dropping {len(dropped)} low-coverage features (< {MIN_FEATURE_COVERAGE:.0%}):')
    for c in dropped:
        print(f'  {c}: {coverage[c]:.0%}')

FEATURE_COLS = [c for c in features_train.columns if c not in dropped]

y_raw = train_df['tp_ratio'].values
X_raw = features_train[FEATURE_COLS].values
valid = ~np.isnan(y_raw)
X, y  = X_raw[valid], y_raw[valid]
TICKERS_TRAIN = train_df['ticker'].values[valid]

print(f'\nML dataset: {X.shape[0]:,} rows × {X.shape[1]} features')
print(f'y (tp_ratio) — mean={y.mean():.3f}, std={y.std():.3f}, range=[{y.min():.3f}, {y.max():.3f}]')
print(f'Feature columns ({len(FEATURE_COLS)}):')
print(FEATURE_COLS)

---
## Cell 11 — Model Cross-Validation

Four models evaluated with 5-fold CV. Each is wrapped in a **MedianImputer → StandardScaler → Model** pipeline.

Progress is shown per model. GradientBoosting typically takes longest (1–3 min with 300 trees).

In [ ]:
def make_pipe(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   model),
    ])

MODELS = {
    'Ridge':            make_pipe(Ridge(alpha=10.0)),
    'Lasso':            make_pipe(Lasso(alpha=0.01, max_iter=5000)),
    'RandomForest':     make_pipe(RandomForestRegressor(
                            n_estimators=300, max_depth=5, min_samples_leaf=10,
                            n_jobs=-1, random_state=RANDOM_STATE)),
    'GradientBoosting': make_pipe(GradientBoostingRegressor(
                            n_estimators=N_ESTIMATORS, max_depth=3,
                            learning_rate=0.03, subsample=0.8,
                            min_samples_leaf=10, random_state=RANDOM_STATE)),
}

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, pipe in tqdm(MODELS.items(), desc='Cross-validating', unit='model'):
    tqdm.write(f'  {name}...')
    r2_scores   = cross_val_score(pipe, X, y, cv=cv, scoring='r2', n_jobs=1)
    mape_scores = cross_val_score(pipe, X, y, cv=cv,
                                  scoring='neg_mean_absolute_percentage_error', n_jobs=1)
    results.append({
        'Model':        name,
        'CV R²':        r2_scores.mean(),
        'CV R² std':    r2_scores.std(),
        'CV MAPE':      -mape_scores.mean(),
    })
    tqdm.write(f'    R²={r2_scores.mean():.3f} ± {r2_scores.std():.3f}   MAPE={-mape_scores.mean():.1%}')

results_df = pd.DataFrame(results).sort_values('CV R²', ascending=False)
print('\nModel comparison:')
print(results_df.round(3).to_string(index=False))

---
## Cell 12 — Model Comparison Chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
order = results_df['Model'].tolist()
x = np.arange(len(order))

ax1.bar(x, results_df['CV R²'],  color='steelblue', alpha=0.8,
        yerr=results_df['CV R² std'], capsize=4)
ax1.set_xticks(x); ax1.set_xticklabels(order, rotation=15)
ax1.set_ylabel('CV R²');  ax1.set_title('Cross-Validated R² (higher = better)')

ax2.bar(x, results_df['CV MAPE'] * 100, color='teal', alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(order, rotation=15)
ax2.set_ylabel('MAPE (%)'); ax2.set_title('Cross-Validated MAPE (lower = better)')

plt.tight_layout()
plt.show()

---
## Cell 13 — Train Best Model on Full Dataset

The best model is retrained on all labeled data.
For GradientBoosting, **warm-start** is used so tqdm can show per-tree progress with a real ETA.
The preprocessor (imputer + scaler) is fit once upfront.

In [ ]:
best_name = results_df.iloc[0]['Model']
print(f'Best model: {best_name}')
print(f'  CV R²={results_df.iloc[0]["CV R²"]:.3f}, MAPE={results_df.iloc[0]["CV MAPE"]:.1%}')

# Fit preprocessor once
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
X_proc = preprocessor.fit_transform(X)

if best_name == 'GradientBoosting':
    gbm = GradientBoostingRegressor(
        n_estimators=1, max_depth=3, learning_rate=0.03,
        subsample=0.8, min_samples_leaf=10,
        warm_start=True, random_state=RANDOM_STATE)
    gbm.fit(X_proc, y)
    with tqdm(total=N_ESTIMATORS, desc='Training GBM', unit='tree') as pbar:
        pbar.update(1)
        for n in range(2, N_ESTIMATORS + 1):
            gbm.set_params(n_estimators=n)
            gbm.fit(X_proc, y)
            if n % 10 == 0:
                pbar.set_postfix(train_loss=f'{gbm.train_score_[-1]:.4f}')
            pbar.update(1)
    final_model = gbm
elif best_name == 'RandomForest':
    final_model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS, max_depth=5, min_samples_leaf=10,
        n_jobs=-1, random_state=RANDOM_STATE)
    with tqdm(total=1, desc='Training RF') as pbar:
        final_model.fit(X_proc, y)
        pbar.update(1)
else:
    final_model = MODELS[best_name].named_steps['model'].__class__(
        **MODELS[best_name].named_steps['model'].get_params())
    final_model.fit(X_proc, y)

# Wrap back into a predict-ready pipeline
best_pipe = Pipeline([
    ('imputer', preprocessor.named_steps['imputer']),
    ('scaler',  preprocessor.named_steps['scaler']),
    ('model',   final_model),
])

y_pred_full = best_pipe.predict(X)
print(f'\nFull-data R²={r2_score(y, y_pred_full):.3f}, MAPE={mean_absolute_percentage_error(y, y_pred_full):.1%}')
print('(Full-data fit is optimistic — use CV scores above for generalization estimate)')

---
## Cell 14 — Feature Importance (Permutation)

Permutation importance shuffles each feature and measures R² drop — works for any model type.

In [ ]:
print('Computing permutation importance (30 repeats)...')
perm = permutation_importance(
    best_pipe, X, y, n_repeats=30,
    random_state=RANDOM_STATE, scoring='r2',
    n_jobs=-1)

imp_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': perm.importances_mean,
    'std':        perm.importances_std,
}).sort_values('importance', ascending=False)

top = imp_df.head(25)
fig, ax = plt.subplots(figsize=(9, max(5, len(top) * 0.38)))
ax.barh(top['feature'], top['importance'],
        xerr=top['std'], color='steelblue', alpha=0.8, capsize=3)
ax.set_xlabel('Mean R² decrease when shuffled')
ax.set_title(f'Feature Importance — {best_name} (permutation, top 25)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 15 features:')
print(imp_df.head(15).round(4).to_string(index=False))

---
## Cell 15 — Residual Analysis

Uses cross-validated predictions (not in-sample) for an honest view of fit quality.

In [ ]:
print('Computing CV predictions for residual analysis...')
y_cv = cross_val_predict(MODELS[best_name], X, y, cv=cv)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.scatter(y, y_cv, alpha=0.3, s=15, color='steelblue')
lims = [min(y.min(), y_cv.min()) - 0.05, max(y.max(), y_cv.max()) + 0.05]
ax1.plot(lims, lims, 'r--', lw=1)
ax1.set_xlabel('Actual tp_ratio'); ax1.set_ylabel('Predicted tp_ratio')
ax1.set_title(f'Actual vs Predicted ({CV_FOLDS}-fold CV)')

residuals = (y_cv - y) / y * 100
ax2.scatter(y, residuals, alpha=0.3, s=15, color='coral')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xlabel('Actual tp_ratio'); ax2.set_ylabel('% Error')
ax2.set_title('Residuals')

worst = np.argsort(np.abs(residuals))[-8:]
for i in worst:
    ax2.annotate(TICKERS_TRAIN[i], (y[i], residuals[i]), fontsize=6, alpha=0.8)

plt.tight_layout(); plt.show()
print(f'CV MAPE={mean_absolute_percentage_error(y, y_cv):.1%}  R²={r2_score(y, y_cv):.3f}')

---
## Cell 16 — SBB / TP Ratio Calibration

The data consistently shows `SBB ≈ 58% of TP` across all stocks and dates.
We calibrate the median ratio and use it as the SBB derivation rule.

In [ ]:
sbb_tp_series = train_df['sbb_tp_ratio'].dropna()
SBB_TP_RATIO  = float(sbb_tp_series.median())

print(f'SBB/TP statistics:')
print(sbb_tp_series.describe().round(4))
print(f'\nCalibrated SBB/TP ratio: {SBB_TP_RATIO:.4f}  (SBB = {SBB_TP_RATIO:.1%} of TP)')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(sbb_tp_series, bins=40, edgecolor='white', color='steelblue', alpha=0.8)
ax.axvline(SBB_TP_RATIO, color='red', linestyle='--', label=f'Median={SBB_TP_RATIO:.3f}')
ax.set_xlabel('SBB / TP'); ax.set_title('SBB/TP Ratio Distribution')
ax.legend(); plt.tight_layout(); plt.show()

# Validate end-to-end: model TP × ratio vs actual SBB
pred_sbb_train   = y_cv * SBB_TP_RATIO * train_df['sheet_price'].values[valid]
actual_sbb_train = train_df['sbb'].values[valid]
sbb_mape = mean_absolute_percentage_error(actual_sbb_train, pred_sbb_train)
print(f'End-to-end CV MAPE for SBB: {sbb_mape:.1%}')

---
## Cell 17 — Predict for All Tickers in Database

Applies the trained model to every ticker in `pe_stats` using the `predict` mode of `build_features`.

In [ ]:
all_stats = get_pe_stats()
print(f'Tickers in DB: {len(all_stats):,}')

features_pred = build_features(all_stats, mode='predict')
for col in FEATURE_COLS:
    if col not in features_pred.columns:
        features_pred[col] = np.nan
X_pred = features_pred[FEATURE_COLS].values

pred_tp_ratio = best_pipe.predict(X_pred)

predictions = all_stats[['ticker', 'current_price']].copy()
predictions['pred_tp_ratio'] = pred_tp_ratio
predictions['pred_tp']       = predictions['current_price'] * pred_tp_ratio
predictions['pred_sbb']      = predictions['pred_tp'] * SBB_TP_RATIO
predictions['pred_upside_pct'] = (pred_tp_ratio - 1) * 100
predictions['buy_signal']    = predictions['current_price'] < predictions['pred_sbb']

print(f'\nPrediction summary:')
print(predictions[['pred_tp_ratio','pred_upside_pct']].describe().round(2))
print(f'\nBuy signals (price < SBB): {predictions["buy_signal"].sum()}')

---
## Cell 18 — Top Buy Opportunities

In [ ]:
buy_df = predictions[predictions['buy_signal']].copy()
buy_df['pct_below_sbb'] = (predictions['pred_sbb'] - predictions['current_price']) \
                           / predictions['current_price'] * 100
buy_df = buy_df.sort_values('pct_below_sbb', ascending=False)

ctx = all_stats[['ticker','market_cap_b','forward_pe','rev_cagr_3yr','current_ps']]
buy_df = buy_df.merge(ctx, on='ticker', how='left')

print(f'Top 30 buy signals:')
print(buy_df[['ticker','current_price','pred_sbb','pred_tp',
              'pred_upside_pct','pct_below_sbb','market_cap_b','forward_pe']]
      .head(30).round(2).to_string(index=False))

---
## Cell 19 — Validate Against Known Spreadsheet Values

Compares model predictions to the analyst's actual SBB and TP for training tickers.

In [ ]:
# Use most recent spreadsheet entry per ticker for comparison
latest = (
    labels.sort_values('sheet_date')
          .groupby('ticker').last()
          .reset_index()[['ticker','sheet_price','sbb','target_price']]
)
val = latest.merge(predictions[['ticker','pred_tp','pred_sbb']], on='ticker', how='inner')
val['tp_pct_err']  = (val['pred_tp']  - val['target_price']) / val['target_price']  * 100
val['sbb_pct_err'] = (val['pred_sbb'] - val['sbb'])          / val['sbb']           * 100

print(f'Validation on {len(val)} tickers (most recent spreadsheet entry):')
print(f'  TP  median |%err|: {val["tp_pct_err"].abs().median():.1f}%')
print(f'  SBB median |%err|: {val["sbb_pct_err"].abs().median():.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pc, ac, title in [
    (axes[0], 'pred_tp',  'target_price', 'Target Price'),
    (axes[1], 'pred_sbb', 'sbb',          'Strong Buy Below'),
]:
    ax.scatter(val[ac], val[pc], alpha=0.5, s=20, color='steelblue')
    lims = [min(val[ac].min(), val[pc].min()) * 0.9,
            max(val[ac].max(), val[pc].max()) * 1.1]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_xlabel(f'Analyst {title}'); ax.set_ylabel(f'Predicted')
    ax.set_title(title)
plt.tight_layout(); plt.show()

print('\nSample (most recent entries):')
print(val[['ticker','sheet_price','sbb','pred_sbb','target_price','pred_tp']]
      .round(2).head(20).to_string(index=False))

---
## Cell 20 — Write Predictions to pe_stats

Adds three columns to `pe_stats` if they don't exist, then writes all predictions.
Re-running overwrites previous values with the latest model output.

In [ ]:
hf_db = HistoricFundamentalsDB(HF_DB_PATH)
conn  = hf_db.conn

for col_def in [
    'pred_tp       DOUBLE',
    'pred_sbb      DOUBLE',
    'pred_model_at TIMESTAMP',
]:
    conn.execute(f'ALTER TABLE pe_stats ADD COLUMN IF NOT EXISTS {col_def}')

now    = datetime.now(timezone.utc)
to_write = predictions.dropna(subset=['pred_tp', 'pred_sbb'])

for _, row in tqdm(to_write.iterrows(), total=len(to_write),
                   desc='Writing to pe_stats', unit='ticker'):
    conn.execute("""
        UPDATE pe_stats
        SET pred_tp = ?, pred_sbb = ?, pred_model_at = ?
        WHERE ticker = ?
    """, [row['pred_tp'], row['pred_sbb'], now, row['ticker']])

print(f'\nWrote predictions for {len(to_write):,} tickers')

sample = conn.execute("""
    SELECT ticker, current_price, pred_sbb, pred_tp, pred_model_at
    FROM pe_stats WHERE pred_tp IS NOT NULL
    ORDER BY ticker LIMIT 8
""").df()
print(sample.round(2).to_string(index=False))
hf_db.close()

---
## Cell 21 — Save Model Artifacts

Three files saved to `models/`:
- `tp_model.joblib` — sklearn Pipeline (imputer → scaler → model)
- `tp_features.joblib` — ordered feature name list
- `sbb_calibration.joblib` — SBB/TP ratio + metadata

**To use outside the notebook:**
```python
import joblib
from historic_fundamentals import get_pe_stats

model    = joblib.load('models/tp_model.joblib')
features = joblib.load('models/tp_features.joblib')
calib    = joblib.load('models/sbb_calibration.joblib')

stats     = get_pe_stats()                          # or a specific ticker list
feat_df   = build_features(stats, mode='predict')   # import build_features from this notebook
X         = feat_df[features].values
pred_tp   = stats['current_price'] * model.predict(X)
pred_sbb  = pred_tp * calib['sbb_tp_ratio']
```

In [ ]:
joblib.dump(best_pipe,   MODEL_FILE)
joblib.dump(FEATURE_COLS, FEATURES_FILE)
joblib.dump({
    'sbb_tp_ratio':  SBB_TP_RATIO,
    'model_name':    best_name,
    'n_training':    int(len(y)),
    'n_tickers':     int(train_df['ticker'].nunique()),
    'cv_r2':         float(results_df.iloc[0]['CV R²']),
    'cv_mape':       float(results_df.iloc[0]['CV MAPE']),
    'trained_at':    datetime.now(timezone.utc).isoformat(),
    'feature_cols':  FEATURE_COLS,
}, CALIB_FILE)

# Sanity check
m2 = joblib.load(MODEL_FILE)
assert list(joblib.load(FEATURES_FILE)) == FEATURE_COLS
assert np.allclose(m2.predict(X[:5]), best_pipe.predict(X[:5]))

print('Saved:')
for f in [MODEL_FILE, FEATURES_FILE, CALIB_FILE]:
    print(f'  {f}  ({Path(f).stat().st_size/1024:.1f} KB)')
print('\nReload verified.')

---
## Cell 22 — Summary

In [ ]:
calib = joblib.load(CALIB_FILE)
print('=' * 60)
print('VALUATION MODEL SUMMARY')
print('=' * 60)
print(f'Model:             {calib["model_name"]}')
print(f'Training rows:     {calib["n_training"]:,}')
print(f'Unique tickers:    {calib["n_tickers"]}')
print(f'CV R²:             {calib["cv_r2"]:.3f}')
print(f'CV MAPE (TP):      {calib["cv_mape"]:.1%}')
print(f'SBB/TP ratio:      {calib["sbb_tp_ratio"]:.4f}  (SBB = {calib["sbb_tp_ratio"]:.1%} of TP)')
print(f'Features:          {len(calib["feature_cols"])}')
print(f'Trained at:        {calib["trained_at"][:19]}')
print()
print('To improve the model:')
print('  1. Add more GP*.xlsx spreadsheets to data/GP/ and re-run')
print('  2. Tune N_ESTIMATORS, max_depth, learning_rate in Cell 2/11')
print('  3. Add features in build_features() in Cell 7')
print('  4. Increase MIN_FEATURE_COVERAGE in Cell 2 to force more complete features')

---
## Cell 23 — Quick Predict (no retraining needed)

Load the saved model and predict TP / SBB for any ticker(s) in the database.

**Minimal setup**: run Cell 1 (imports) and Cell 2 (config), then run this cell.  
No spreadsheets, no training loop required.

In [ ]:
import joblib, numpy as np, pandas as pd
from historic_fundamentals import get_pe_stats

_model    = joblib.load(MODELS_DIR / 'tp_model.joblib')
_features = joblib.load(MODELS_DIR / 'tp_features.joblib')
_calib    = joblib.load(MODELS_DIR / 'sbb_calibration.joblib')

def _sr(a, b, clip=20.0):
    return (a / b.replace(0, np.nan)).clip(-clip, clip)

def predict_ticker(tickers):
    """
    Predict TP and SBB for one or more tickers using the saved model.

    Args:
        tickers: str or list[str]

    Returns:
        DataFrame with: ticker, current_price, pred_tp, pred_sbb, pred_upside_pct, buy_signal
    """
    if isinstance(tickers, str):
        tickers = [tickers]

    stats = get_pe_stats(tickers)
    if stats.empty:
        print(f'No fundamentals found for: {tickers}')
        return pd.DataFrame()

    missing = set(t.upper() for t in tickers) - set(stats['ticker'])
    if missing:
        print(f'Not in database (not yet imported): {sorted(missing)}')

    def _col(name):
        return stats[name] if name in stats.columns else pd.Series(np.nan, index=stats.index)

    f = pd.DataFrame(index=stats.index)
    f['f_pe']             = _col('current_pe')
    f['f_ps']             = _col('current_ps')
    f['f_ptbv']           = _col('current_ptbv')
    f['f_ev_ebitda']      = _col('current_evebitda')
    f['f_fcf_yield']      = _col('current_fcf_yield')
    f['f_div_yield']      = _col('dividend_yield')
    f['f_roce']           = _col('current_roa')
    f['f_log_mktcap']     = np.log1p(_col('market_cap_b').clip(lower=0))
    f['f_tps']            = np.nan
    f['f_vintage_months'] = 0.0

    f['f_pe_premium']     = _sr(f['f_pe'],        _col('pe_lt_median'))
    f['f_ps_premium']     = _sr(f['f_ps'],         _col('ps_lt_median'))
    f['f_ptbv_premium']   = _sr(f['f_ptbv'],       _col('ptbv_lt_median'))
    f['f_ev_premium']     = _sr(f['f_ev_ebitda'],  _col('evebitda_lt_median'))

    f['f_pe_lt_med']      = _col('pe_lt_median')
    f['f_ps_lt_med']      = _col('ps_lt_median')
    f['f_pfcf_lt_med']    = _col('pfcf_lt_median')
    f['f_ev_lt_med']      = _col('evebitda_lt_median')
    f['f_ptbv_lt_med']    = _col('ptbv_lt_median')
    f['f_pe_5yr_med']     = _col('pe_rolling_5yr_median')
    f['f_ps_5yr_med']     = _col('ps_rolling_5yr_median')
    f['f_forward_pe']     = _col('forward_pe')

    f['f_rev_g1']         = _col('rev_growth_1yr')
    f['f_rev_c3']         = _col('rev_cagr_3yr')
    f['f_rev_c5']         = _col('rev_cagr_5yr')
    f['f_rev_ntm']        = _col('rev_ntm_growth_est')
    f['f_earn_g1']        = _col('earn_growth_1yr')
    f['f_earn_c3']        = _col('earn_cagr_3yr')
    f['f_earn_c5']        = _col('earn_cagr_5yr')
    f['f_earn_ntm']       = _col('earn_ntm_growth_est')
    f['f_fcf_g1']         = _col('fcf_growth_1yr')
    f['f_fcf_c3']         = _col('fcf_cagr_3yr')
    f['f_fcf_c5']         = _col('fcf_cagr_5yr')
    f['f_fcf_margin']     = _col('fcf_margin_5yr_median')
    f['f_ebitda_margin']  = _col('ebitda_margin_5yr_median')
    f['f_roic_lt']        = _col('roic_lt_median')
    f['f_roe_lt']         = _col('roe_lt_median')
    f['f_roa_lt']         = _col('roa_lt_median')
    f['f_months_hist']    = _col('months_available')

    for col in _features:
        if col not in f.columns:
            f[col] = np.nan

    tp_ratio  = _model.predict(f[_features].values)
    sbb_ratio = _calib['sbb_tp_ratio']

    out = stats[['ticker', 'current_price']].copy()
    out['pred_tp']         = (out['current_price'] * tp_ratio).round(2)
    out['pred_sbb']        = (out['pred_tp'] * sbb_ratio).round(2)
    out['pred_upside_pct'] = ((tp_ratio - 1) * 100).round(1)
    out['buy_signal']      = out['current_price'] < out['pred_sbb']
    return out.reset_index(drop=True)


print(f'Model loaded: {_calib["model_name"]}')
print(f'Trained on {_calib["n_training"]:,} rows  |  CV R²={_calib["cv_r2"]:.3f}  |  MAPE={_calib["cv_mape"]:.1%}')
print(f'SBB/TP ratio: {_calib["sbb_tp_ratio"]:.3f}')
print()

# --- change tickers here ---
predict_ticker(['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'META'])